### Download the negative tsv file from Uniprot using the Request API

In [1]:
import requests
import pandas as pd

url = "https://rest.uniprot.org/uniprotkb/stream"  

# parameters and structure of the query
params = {
    "query": "(existence:1) AND (reviewed:true) AND (fragment:false) AND (taxonomy_id:2759) "
             "AND (length:[40 TO *]) AND "
             "((cc_scl_term_exp:SL-0091) OR (cc_scl_term_exp:SL-0191) OR "
             "(cc_scl_term_exp:SL-0173) OR (cc_scl_term_exp:SL-0209) OR "
             "(cc_scl_term_exp:SL-0204) OR (cc_scl_term_exp:SL-0039)) "
             "NOT (ft_signal:*)",          
    "fields": "accession,organism_name,reviewed,id,length,ft_transmem,lineage,sequence",  
    "format": "tsv",   
}
r = requests.get(url, params=params, timeout=300)
r.raise_for_status()

# create the tsv file
with open("uniprot_results.tsv", "w") as f:
    f.write(r.text)

# convert the tsv file into a dataframe
df = pd.read_csv("uniprot_results.tsv", sep="\t")


In [2]:
# check the dataframe
print(df.info)
df.columns

<bound method DataFrame.info of             Entry                                           Organism  \
0      A0A061ACU2                             Caenorhabditis elegans   
1      A0A061AE05                             Caenorhabditis elegans   
2      A0A075D657                    Vinca minor (Common periwinkle)   
3      A0A075TRC0        Penicillium expansum (Blue mold rot fungus)   
4      A0A076FFM5                     Ocimum basilicum (Sweet basil)   
...           ...                                                ...   
20969      V5IQE0  Neurospora crassa (strain ATCC 24698 / 74-OR23...   
20970      C0HMD7                               Homo sapiens (Human)   
20971      P19407                        Spinacia oleracea (Spinach)   
20972      P40166  Saccharomyces cerevisiae (strain ATCC 204508 /...   
20973      P53307  Saccharomyces cerevisiae (strain ATCC 204508 /...   

       Reviewed   Entry Name  Length  \
0      reviewed  PIEZ1_CAEEL    2442   
1      reviewed  PAPSH_

Index(['Entry', 'Organism', 'Reviewed', 'Entry Name', 'Length',
       'Transmembrane', 'Taxonomic lineage', 'Sequence'],
      dtype='str')

### Filter the entries of the negative set 

In [3]:
import re
import pandas as pd

# this function checks if the transmembrane helics startpoint starts within the first 90 residues
def tm_starts(s):
    if pd.isna(s):
        return []
    return [int(x) for x in re.findall(r'TRANSMEM\s+[<>?]?(\d+)', str(s))]

# theese two functions split the Taxonomic lineage field into kingdom and organism
def kingdom(s):
    parts = [p.strip() for p in str(s).split(',')]
    names = {re.sub(r'\s*\(.*\)$', '', p) for p in parts}
    if 'Metazoa' in names:         return 'Metazoa'
    if 'Fungi' in names:           return 'Fungi'
    if 'Viridiplantae' in names:   return 'Plants'
    return 'Other'


# create the dataframe with the filtered data
filtered_df = pd.DataFrame({
    'accession':      df['Entry'],
    'organism':       df['Organism'],
    'kingdom':        df['Taxonomic lineage'].apply(kingdom),
    'length':         df['Length'].astype(int),
    'tm_in_first_90': df['Transmembrane'].apply(lambda s: any(p <= 90 for p in tm_starts(s))),
    'sequence':       df['Sequence'],
})

# convert the dataframe into a tsv file
filtered_df.to_csv('negative_data.tsv', sep='\t', index=False)

In [4]:
# check the new df
print(filtered_df.head(10))
print()
print(filtered_df['tm_in_first_90'].value_counts())

    accession                                           organism  kingdom  \
0  A0A061ACU2                             Caenorhabditis elegans  Metazoa   
1  A0A061AE05                             Caenorhabditis elegans  Metazoa   
2  A0A075D657                    Vinca minor (Common periwinkle)   Plants   
3  A0A075TRC0        Penicillium expansum (Blue mold rot fungus)    Fungi   
4  A0A076FFM5                     Ocimum basilicum (Sweet basil)   Plants   
5  A0A078CGE6                              Brassica napus (Rape)   Plants   
6  A0A087WPF7                               Mus musculus (Mouse)  Metazoa   
7  A0A087X1C5                               Homo sapiens (Human)  Metazoa   
8  A0A095C325  Cryptococcus deuterogattii (strain R265) (Cryp...    Fungi   
9  A0A096LP01                               Homo sapiens (Human)  Metazoa   

   length  tm_in_first_90                                           sequence  
0    2442            True  MTVPPLLKSCVVKLLLPAALLAAAIIRPSFLSIGYVLLALVSAVLP

### Create filtered fasta file

In [5]:
with open("negative_SP_clean.fasta", "w") as f:
    for _, row in df.iterrows():
        f.write(f">{row['Entry']}\n{row['Sequence']}\n")   